<a href="https://colab.research.google.com/github/dhana0020/Deep-Learning/blob/main/LSTM_AND_GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import re
from collections import Counter
import numpy as np
from datasets import load_dataset

In [ ]:
dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

print(train_data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
def clean_text(text):
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = text.lower()
    return text

In [ ]:
def build_vocab(data, max_vocab=20000):
    counter = Counter()

    for item in data:
        text = clean_text(item["text"])
        counter.update(text.split())

    vocab = {"<PAD>":0, "<UNK>":1}
    most_common = counter.most_common(max_vocab-2)

    for word, _ in most_common:
        vocab[word] = len(vocab)

    return vocab

vocab = build_vocab(train_data)
vocab_size = len(vocab)
print("Vocab size:", vocab_size)

Vocab size: 20000


In [ ]:
MAX_LEN = 200

def encode(text):
    text = clean_text(text)
    tokens = text.split()
    ids = [vocab.get(word, vocab["<UNK>"]) for word in tokens]
    ids = ids[:MAX_LEN]
    ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))
    return ids

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, data):
        self.texts = [encode(item["text"]) for item in data]
        self.labels = [item["label"] for item in data]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx]), torch.tensor(self.labels[idx])

In [ ]:
train_dataset = IMDBDataset(train_data)
test_dataset = IMDBDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(LSTMModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return self.sigmoid(out)

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(GRUModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.gru(x)
        out = self.fc(hidden[-1])
        return self.sigmoid(out)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embed_dim = 128
hidden_dim = 128

lstm_model = LSTMModel(vocab_size, embed_dim, hidden_dim).to(device)
gru_model = GRUModel(vocab_size, embed_dim, hidden_dim).to(device)

criterion = nn.BCELoss()
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)
optimizer_gru = optim.Adam(gru_model.parameters(), lr=0.001)

In [ ]:
def train(model, loader, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(loader)

    for texts, labels in loop:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts).squeeze()
        loss = criterion(outputs, labels.float())

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (outputs > 0.5).int()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=loss.item())

    return total_loss / len(loader), correct / total

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0

    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)

            outputs = model(texts).squeeze()
            loss = criterion(outputs, labels.float())

            preds = (outputs > 0.5).int()

            total_loss += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1} =====")

    loss_lstm, train_acc_lstm = train(lstm_model, train_loader, optimizer_lstm)
    test_acc_lstm = evaluate(lstm_model, test_loader)


    print(f"\nLSTM  -> Loss: {loss_lstm:.4f}, Train Acc: {train_acc_lstm:.4f}, Test Acc: {test_acc_lstm:.4f}")



===== Epoch 1 =====


100%|██████████| 391/391 [00:05<00:00, 65.61it/s, loss=0.691]



LSTM  -> Loss: 0.6939, Train Acc: 0.5120, Test Acc: 0.5005

===== Epoch 2 =====


100%|██████████| 391/391 [00:04<00:00, 84.04it/s, loss=0.698]



LSTM  -> Loss: 0.6893, Train Acc: 0.5233, Test Acc: 0.5080

===== Epoch 3 =====


100%|██████████| 391/391 [00:04<00:00, 79.79it/s, loss=0.644]



LSTM  -> Loss: 0.6638, Train Acc: 0.5852, Test Acc: 0.7249

===== Epoch 4 =====


100%|██████████| 391/391 [00:04<00:00, 81.28it/s, loss=0.371]



LSTM  -> Loss: 0.4162, Train Acc: 0.8169, Test Acc: 0.8266

===== Epoch 5 =====


100%|██████████| 391/391 [00:04<00:00, 78.55it/s, loss=0.264]



LSTM  -> Loss: 0.2664, Train Acc: 0.8955, Test Acc: 0.8377


In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1} =====")



    loss_gru, train_acc_gru = train(gru_model, train_loader, optimizer_gru)
    test_acc_gru = evaluate(gru_model, test_loader)


    print(f"GRU   -> Loss: {loss_gru:.4f}, Train Acc: {train_acc_gru:.4f}, Test Acc: {test_acc_gru:.4f}")


===== Epoch 1 =====


100%|██████████| 391/391 [00:05<00:00, 75.77it/s, loss=0.682]


GRU   -> Loss: 0.6941, Train Acc: 0.5174, Test Acc: 0.5546

===== Epoch 2 =====


100%|██████████| 391/391 [00:05<00:00, 77.36it/s, loss=0.687]


GRU   -> Loss: 0.6647, Train Acc: 0.5968, Test Acc: 0.6477

===== Epoch 3 =====


100%|██████████| 391/391 [00:04<00:00, 80.80it/s, loss=0.313]


GRU   -> Loss: 0.4825, Train Acc: 0.7744, Test Acc: 0.8140

===== Epoch 4 =====


100%|██████████| 391/391 [00:04<00:00, 79.65it/s, loss=0.221]


GRU   -> Loss: 0.3013, Train Acc: 0.8752, Test Acc: 0.8461

===== Epoch 5 =====


100%|██████████| 391/391 [00:04<00:00, 83.84it/s, loss=0.265]


GRU   -> Loss: 0.2105, Train Acc: 0.9207, Test Acc: 0.8542


In [ ]:
lstm_loss, lstm_acc = evaluate(lstm_model, test_loader, criterion)

print("LSTM Test Loss:", lstm_loss)
print("LSTM Test Accuracy:", lstm_acc)

LSTM Test Loss: 0.3819936882237644
LSTM Test Accuracy: 0.83768


In [ ]:
gru_loss, gru_acc = evaluate(gru_model, test_loader, criterion)

print("GRU Test Loss:", gru_loss)
print("GRU Test Accuracy:", gru_acc)

GRU Test Loss: 0.3494841133999398
GRU Test Accuracy: 0.85424


In [ ]:
def predict(model, sentence):
    model.eval()

    encoded = encode(sentence)
    tensor = torch.tensor([encoded]).to(device)

    with torch.no_grad():
        output = model(tensor)
        prob = output.item()

    if prob > 0.5:
        return "Positive 😊"
    else:
        return "Negative 😠"

In [ ]:
print(predict(lstm_model, "This movie was absolutely fantastic!"))
print(predict(gru_model, "I hated this movie. It was boring."))

Positive 😊
Negative 😠



GRU achieved the highest accuracy (85.42%).

LSTM achieved slightly lower accuracy (83.77%).

RNN achieved significantly lower accuracy (70.17%).

Accuracy Ranking:
GRU > LSTM >> RNN

The difference between GRU and LSTM is small (~1.6%), but both clearly outperform RNN.